Loaddocs

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import  GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq
import os



In [6]:
file_path2 = "RAG_SYSTEMS/medical_report.pdf"
loader = PyPDFLoader(file_path2)
docs = loader.load()

In [8]:
splitter= RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splited_docs=splitter.split_documents(docs)

len(splited_docs)


26

In [12]:
from google.genai import documents
from filetype.types import document
key1=os.getenv("Gemini_KEY")
embeddings=GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    google_api_key=key1
)

vector_store=InMemoryVectorStore.from_documents(
    documents=splited_docs,
    embedding=embeddings
)


In [25]:
def retrive_tool(query1:str):
    """
    This tools will help you to retrive datat of relevant document and these documents have detail about medical reports
    """
    docs=vector_store.similarity_search(query=query1,k=4)
    content=""
    for doc in docs:
        content=doc.page_content+"\n\n"
    return content

In [26]:
retrive_tool("Patient")

'Report Status    \nFemale\n27 Years:\n:\n:\n:\nAge\nGender\nReported        \nP\n09/07/2025  04:56:00PM\nDR NITIN NAHAR\n474764803\nMs. NIKITA  CHUDHARY:\n:\n:\n:\n:\nName        \nLab No.    \nRef By \nCollected       \nA/c Status Final\n10/7/2025  6:31:50PM\nCollected at            : Processed at             :BHOPAL CC-82\nMr Rachel V John Pata So Vitus John Mig 26 \nGraund,Indrapuri, Phone: 8770817968\n \nLPL - Bhopal Lab II\nPlot No.05, Mandakini Housing Society, \nNear Apurti Shopping Mall, Kolar Main \nRoad, Bhopal, M.P. -  462042\nTest Report \nTest Name Results Units Bio. Ref. Interval\n4.60Albumin  3.50 - 5.20 g/dL\n2.89Globulin(Calculated)  2.0 - 3.5 gm/dL\n1.59A : G Ratio  0.90 - 2.00 \n9.92Calcium, Total  8.60 - 10.30 mg/dL\n4.34Phosphorus  2.40 - 4.40 mg/dL\n137.31Sodium  136.00 - 146.00 mEq/L\n4.86Potassium  3.50 - 5.10 mEq/L\n103.66Chloride  101.00 - 109.00 mEq/L\nNote\n1. Estimated GFR  (eGFR) calculated using the 2021 CKD-EPI creatinine equation and GFR Category repor

In [19]:
key2=os.getenv("GroqKEY")
llm1=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=key2)


In [20]:
prompt1="""
    you are a helpful assistant that answers questions using RAG
    always use 'retrive_tool' for questions requiring external knowledge.
"""

In [23]:
from langchain.agents import create_agent


agennt1=create_agent(
    model=llm1,
    tools=[retrive_tool],
    system_prompt=prompt1
)

In [28]:

query=input("User: ")
res=agennt1.invoke({"messages":[{"role":"user","content":query}]})
# print()
# # print(res["messages"][-1].content)
    